

##LangChain & HAG Project

Projeto técnico focado na implementação do framework LangChain e arquitetura HAG (Hybrid Augmented Generation) para otimização de LLMs.

🛠️ Stack Técnica
Orquestração: LangChain

Arquitetura: HAG (Recuperação Híbrida)

Dados: Vetorização e Busca Semântica

🏗️ Pipeline de Dados
Ingestão: Carregamento e Chunking (fatiamento) inteligente de documentos.

Indexação: Armazenamento em Vector Database para recuperação eficiente.

RAG/HAG: Enriquecimento de prompt com contexto recuperado para mitigar alucinações e aumentar a precisão da resposta.

RAG (Retrieval-Augmented Generation): É uma técnica/arquitetura que une o modelo de IA a uma base de dados externa. Em vez de a IA responder apenas com o conhecimento com o qual foi treinada, o sistema busca documentos relevantes (Retrieval) e os entrega para a IA criar uma resposta precisa e atualizada (Generation), evitando alucinações.

LangChain: É o framework (a ferramenta de desenvolvimento) que serve para orquestrar e conectar as peças de um sistema de IA. É ele quem liga o modelo de linguagem (LLM) aos seus bancos de dados, APIs, ferramentas de memória e fluxos de trabalho.

Embeddings: É o processo de transformar textos em números (vetores) de forma que o computador entenda o significado das palavras. Duas frases com significados parecidos geram números parecidos, permitindo que o sistema faça buscas por contexto e significado, e não apenas por palavras-chave exatas.

### Bibliotecas


In [0]:
!pip install langchain
!pip install langchain langchain-community

In [0]:
import langchain
from langchain_community.document_loaders import TextLoader

In [0]:
##Conf variavel ambiente 
OPENAI_API_KEY = "ccc"

### Vetorizar com Embeddings

In [0]:
###Leitura de dados txt
## Arquivo formado por metadados e texto 
documento = TextLoader("/Workspace/Repos/patriciatamiresdesousa@gmail.com/LangChain_Tecnicas_Avancadas-_RAG/GTB_gold_Nov23.txt", encoding="utf-8").load()
documento

##### DOC = CHUNKING
- Fatiar documentos grandes em pedaços menores para não estourar o limite de leitura da IA
- Aumentar a precisão da busca, permitindo encontrar o parágrafo exato que responde à dúvida do usuário
- Economizar custos e evitar confusão, enviando para o prompt apenas a informação estritamente necessária

In [0]:
!pip install langchain-text-splitters

In [0]:
## Aqui estou fazendo um chuck meio bruto, mas o ideial é usar alumas tecnicas mais avancadas
## isso porque ele vai escolher as primeiras mil linhas do docuemntos e armazenar como um documento, para evitar que falte palavras como por exemplo (arroz ele pegar apenas arro eu uso chunk_overlap para evitar esse erro

from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
pedacos = splitter.split_documents(documento)

In [0]:
##pegou meu texto e dividiu em 5 pedacos
len(pedacos)

In [0]:
### Depois do chuck, eu uso embedding para converte em vetores as informações sendo a forma mais eficaz de fazer buscas em textos, diferente por exemplo do sql onde eu coloco like e a busca é em cima daquela palavra
## com embedding eu consigo fazer uma busca mais eficaz, por exemplo se eu quero buscar por arroz, ele vai buscar por arroz, arroz branco, arroz integral, arroz preto, arroz com leite) criando em cima de cada chuck uma matriz vetorial de números para buscar as informações

In [0]:
from langchain_openai import OpenAIEmbeddings

embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-small", 
    openai_api_key=OPENAI_API_KEY
)

In [0]:
## Exemplo de vetor que armazeno as informações do meu texto 
embeddings_model.embed_query(pedacos[0].page_content)

In [0]:
## Aqui eu vou criar um banco de dados para os meus vetores, em memoria mesmo sem documentos
##outro ponto é a escolha qual modelo de embeddings usar porque acima eu só escolhi um modelo geerico para teste
from langchain_community.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore.from_documents(
    documents=pedacos, 
    embedding=embeddings_model
)

In [0]:
### construo um buscador pra pode usar esse banco vetorial
##Então dentro do meu conjunto de chuck eu tenho 5 conjuntos de vetores, aqui eu escolhi 2 e busquei dentro desse 2 vetores onde tem a palavras Seguro Viagem

retriever = vector_store.as_retriever(search_kwargs={"k": 2})
retriever.invoke("Seguro Viagem")

In [0]:
similar_chunks = retriever.invoke("query")
similar_chunks

In [0]:
###Teste de consulta para entender qual vetor é associado a essa pergunta e ele me retora isso

query = "Como devo proceder caso tenha um item roubado?"
query_embed = embeddings_model.embed_query(query)
query_embed

In [0]:
###Logica nessa sequencia (index, embedding, text chuck, metadata)

In [0]:
### Regastar os documentos similares ao banco de dados 

similar_texts = [pedacos.page_content for pedacos in similar_chunks]
similar_texts

In [0]:
### Aumntar o prompt com consulta e documentos

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
 [ ("system", "Responda usando exclusivamente os conteudos fornecidos. \n\nContexto:\n{context}"),
   ("human", "{query}")
 ]

)

In [0]:
## Gerar respostas


from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

modelo = ChatOpenAI(model="gpt-4.1-nano", temperature=0.2, openai_api_key=OPENAI_API_KEY)

In [0]:
modelo.invoke(query)

In [0]:
cadeia = prompt | modelo | StrOutputParser()
trechos = retriever.invoke(query)

contexto = "\n\n".join(trecho.page_content for trecho in trechos)
cadeia.invoke({"query": query, "context": contexto})


####Debug com o LangSmith

In [0]:
pip install --upgrade langchain langchain-core langchain-community

In [0]:
#### Objetivo aqui é entender o de trás do framework. então aqui eu abro arquitetura do lagchain para avaliar cmo isso funciona
### Monstra a cadeia que foi chamada

from langchain_core.globals import set_debug, set_verbose
set_debug(True)
cadeia.invoke({"query": query, "context": contexto})